# Практическая работа №3: Обработка выборочных данных. Нахождение интервальных оценок параметров распределения. Проверка статистической гипотезы о нормальном распределении

Выполнили студенты гр. 2381 Дудкин Михаил Валерьевич и Газукина Дарья Денисовна. Вариант №19.

## Цель работы

Получение практических навыков вычисления интервальных статистических оценок параметров распределения выборочных данных и проверки «справедливости» статистических гипотез.

## Основные теоретические положения

**Интервальная статистическая оценка** — оценка неизвестного параметра генеральной совокупности в виде интервала, который с заданной вероятностью накрывает истинное значение параметра:
$$
\hat{\theta}_1(X_1, \dots, X_n) < \theta < \hat{\theta}_2(X_1, \dots, X_n)
$$
где $\hat{\theta}_1$ и $\hat{\theta}_2$ — случайные величины, зависящие от выборки.

**Статистическая гипотеза** — предположение о виде неизвестного распределения или о параметрах известного распределения, которое проверяется на основе выборочных данных:
$$
H_0: \theta = \theta_0 \quad \text{(нулевая гипотеза)}, \qquad H_1: \theta \neq \theta_0 \quad \text{(альтернативная гипотеза)}
$$

**Надежность интервальной оценки (доверительная вероятность)** — вероятность $\gamma$ того, что построенный доверительный интервал накроет истинное значение оцениваемого параметра:
$$
\gamma = P\left(\hat{\theta}_1 < \theta < \hat{\theta}_2\right), \quad \gamma \in (0, 1)
$$
На практике обычно используют $\gamma = 0.95$ или $\gamma = 0.99$.

**Доверительный интервал** — интервал $(\hat{\theta}_1; \hat{\theta}_2)$, который с заданной надежностью $\gamma$ накрывает неизвестный параметр $\theta$. Для математического ожидания при неизвестном $\sigma$:
$$
\bar{x} - t_{\gamma, n-1} \cdot \frac{s}{\sqrt{n}} < \mu < \bar{x} + t_{\gamma, n-1} \cdot \frac{s}{\sqrt{n}}
$$
где $t_{\gamma, n-1}$ — квантиль распределения Стьюдента, $s$ — выборочное среднеквадратичное отклонение.

Для среднеквадратичного отклонения $\sigma$:
$$
s \cdot \sqrt{\frac{n}{\chi^2_{1-\alpha/2}}} < \sigma < s \cdot \sqrt{\frac{n}{\chi^2_{\alpha/2}}}
$$
где $\chi^2_{\alpha/2}$, $\chi^2_{1-\alpha/2}$ — квантили $\chi^2$-распределения с $n-1$ степенями свободы.

**Критерий Пирсона $\chi^2$** — статистический критерий для проверки гипотезы о согласии эмпирического распределения с теоретическим. Наблюдаемое значение критерия:
$$
\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{(n_i - n'_i)^2}{n'_i} = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - n
$$
где $n_i$ — эмпирические частоты, $n'_i = n \cdot p_i$ — теоретические частоты, $k$ — число интервалов.

Гипотеза $H_0$ отвергается, если $\chi^2_{\text{набл}} > \chi^2_{\text{крит}}(\alpha, df)$, где $df = k - r - 1$ ($r$ — число оценённых параметров).

**Ошибки первого и второго рода** — возможные ошибки при проверке статистических гипотез:

| Ошибка | Условие |  Вероятность                                      |
|--------|---------|---------------------------------------------------|
| **Первого рода** | Отклонить $H_0$, когда она верна | $\alpha=P(\text{отклонить } H_0 \mid H_0 \text{ верна})$ |
| **Второго рода** | Принять $H_0$, когда верна $H_1$ | $\beta=P(\text{принять } H_0 \mid H_1 \text{ верна})$ |

**Мощность критерия** — вероятность правильно отвергнуть ложную нулевую гипотезу:
$$
\text{Мощность} = 1 - \beta
$$
Чем выше мощность, тем чувствительнее критерий к отклонениям от $H_0$.

## Постановка задачи

Для заданной надежности определить (на основании выборочных данных и результатов выполнения практической работы №2) границы доверительных интервалов для математического ожидания и среднеквадратичного отклонения случайной величины. Проверить гипотезу о нормальном распределении исследуемой случайной величины с помощью критерия Пирсона $\chi^2$. Дать содержательную интерпретацию полученным результатам.

## Выполнение работы

Импорт необходимых библиотек

In [89]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as st

Загрузка выборки

In [90]:
# Вариант №19
N = 103   # Объем выборки

In [91]:
path = "/content/sample_103.csv"
df = pd.read_csv(path)
sample = df['nu'].values

# Задание 1

Вычислить точность и доверительный интервал для математического ожидания при неизвестном среднеквадратичном отклонении при заданном объёме выборки для доверительной точности $γ∈\{0.95,0.99\}$. Сделать выводы.

### Доверительный интервал для $\mu$ при неизвестном $\sigma$

**Исходные предположения:**
*   $X \sim N(\mu, \sigma^2)$ — генеральная совокупность имеет нормальное распределение
*   Выборка: $x_1, x_2, \dots, x_N$, параметры $\mu, \sigma$ неизвестны

**Оценки по выборке:**
$$
\bar{x} = \frac{1}{N}\sum_{i=1}^{N} x_i, \quad
s = \sqrt{\frac{1}{N-1}\sum_{i=1}^{N} (x_i - \bar{x})^2}
$$

**Пивотальная величина:**
Так как $\sigma$ неизвестно:

$$
T = \frac{\bar{x} - \mu}{s/\sqrt{N}} \sim t_{N-1}
$$


**Построение интервала:**
Для доверительной вероятности $\gamma = 1 - \alpha$:
$$
P\left(-t_{\alpha/2, N-1} < \frac{\bar{x} - \mu}{s/\sqrt{N}} < t_{\alpha/2, N-1}\right) = \gamma
$$

Разрешая относительно $\mu$:
$$
\mu \in \left(\bar{x} \pm t_{\alpha/2, N-1} \cdot \frac{s}{\sqrt{N}}\right)
$$

где $t_{\alpha/2, N-1}$ — критическое значение $t$-распределения с $N-1$ степенями свободы.


In [92]:
x_bar = np.mean(sample)     # Выборочное среднее
s = np.std(sample, ddof=1)  # Исправленное выборочное СКО

print(f"Объем выборки N = {N}")
print(f"Выборочное среднее = {x_bar:.4f}")
print(f"Выборочное СКО = {s:.4f}\n")

for gamma in [0.95, 0.99]:
    alpha = 1 - gamma
    df = N - 1
    t_crit = st.t.ppf(1 - alpha/2, df)
    delta = t_crit * s / np.sqrt(N)

    lower_bound = x_bar - delta
    upper_bound = x_bar + delta

    print(f"Доверительная вероятность gamma = {gamma}")
    print(f"Уровень значимости alpha = {alpha:.2f}")
    print(f"Критическое значение t = {t_crit:.4f}")
    print(f"Точность оценки delta = {delta:.4f}")
    print(f"Доверительный интервал: ({lower_bound:.4f}; {upper_bound:.4f})\n")

Объем выборки N = 103
Выборочное среднее = 454.0000
Выборочное СКО = 54.3706

Доверительная вероятность gamma = 0.95
Уровень значимости alpha = 0.05
Критическое значение t = 1.9835
Точность оценки delta = 10.6262
Доверительный интервал: (443.3738; 464.6262)

Доверительная вероятность gamma = 0.99
Уровень значимости alpha = 0.01
Критическое значение t = 2.6249
Точность оценки delta = 14.0623
Доверительный интервал: (439.9377; 468.0623)



**Полученные результаты:**
*   При $\gamma = 0.95$: $\mu \in (443.37; 464.63)$
*   При $\gamma = 0.99$: $\mu \in (439.94; 468.06)$

С увеличением доверительной вероятности с $0.95$ до $0.99$ ширина доверительного интервала увеличивается с $21.25$ до $28.13$. Это отражает компромисс между надёжностью оценки и её точностью: чтобы гарантировать попадание истинного $\mu$ в интервал с большей вероятностью, необходимо расширить границы.

С доверительной вероятностью $0.95$ истинное математическое ожидание генеральной совокупности находится в интервале $(443.37; 464.63)$. При повышении надёжности до $0.99$ интервал расширяется до $(439.94; 468.06)$.

# Задание 2

Для вычисления границ доверительного интервала для среднеквадратичного отклонения определить значение $q$ при заданных $γ$ и $n$. Построить доверительные интервалы, сделать выводы.

### Доверительный интервал для $\sigma$ при неизвестном $\mu$

При нормальном распределении и неизвестном $\mu$ статистика
$$
\frac{(N-1)s^2}{\sigma^2} \sim \chi^2_{N-1}
$$
имеет хи-квадрат распределение с $N-1$ степенями свободы.

**Построение интервала для $\sigma^2$:**
Для доверительной вероятности $\gamma = 1 - \alpha$:
$$
P\left(\chi^2_{1-\alpha/2, N-1} < \frac{(N-1)s^2}{\sigma^2} < \chi^2_{\alpha/2, N-1}\right) = \gamma
$$

Разрешая относительно $\sigma^2$:
$$
\frac{(N-1)s^2}{\chi^2_{\alpha/2, N-1}} < \sigma^2 < \frac{(N-1)s^2}{\chi^2_{1-\alpha/2, N-1}}
$$

**Итоговая формула для $\sigma$:**
$$
\sigma \in \left(s \cdot \sqrt{\frac{N-1}{\chi^2_{\alpha/2, N-1}}},\; s \cdot \sqrt{\frac{N-1}{\chi^2_{1-\alpha/2, N-1}}}\right)
$$

где:
* $\chi^2_{\alpha/2, N-1}$ — верхний $\alpha/2$-квантиль $\chi^2$-распределения,
* $\chi^2_{1-\alpha/2, N-1}$ — нижний $\alpha/2$-квантиль.

In [93]:
print(f"N = {N}, s = {s:.4f}\n")

for gamma in [0.95, 0.99]:
    alpha = 1 - gamma
    df = N - 1

    chi2_lower = st.chi2.ppf(alpha/2, df)      # нижний квантиль
    chi2_upper = st.chi2.ppf(1 - alpha/2, df)  # верхний квантиль

    # Границы интервала
    sigma_lower = s * np.sqrt(df / chi2_upper)
    sigma_upper = s * np.sqrt(df / chi2_lower)

    print(f"gamma = {gamma}: X^2_крит = [{chi2_lower:.3f}, {chi2_upper:.3f}]")
    print(f"Интервал для sigma: ({sigma_lower:.4f}; {sigma_upper:.4f})\n")

N = 103, s = 54.3706

gamma = 0.95: X^2_крит = [75.946, 131.838]
Интервал для sigma: (47.8238; 63.0104)

gamma = 0.99: X^2_крит = [68.965, 142.532]
Интервал для sigma: (45.9947; 66.1224)



**Полученные результаты:**
*   При $\gamma = 0.95$: $\sigma \in (47.82; 63.01)$
*   При $\gamma = 0.99$: $\sigma \in (46.00; 66.12)$

Точечная оценка $s = 54.37$ не является серединой доверительного интервала, что обусловлено асимметрией $\chi^2$-распределения. С ростом доверительной вероятности границы интервала расширяются: ширина интервала увеличивается с $15.19$ ($\gamma=0.95$) до $20.13$ ($\gamma=0.99$), что соответствует ожидаемому компромиссу между надёжностью и точностью оценки.

Полученные интервалы охватывают значение $s$ и имеют приемлемую относительную ширину (~28–37% от $s$), что объясняется достаточным объёмом выборки ($N = 103$). Важно отметить, что построенные интервалы справедливы только при выполнении предположения о нормальности генеральной совокупности, которое будет проверено в следующем задании.

# Задания 3-5

Проверить гипотезу о нормальности заданного распределения с помощью критерия $\chi^2$ (Пирсона). Для этого необходимо найти теоретические частоты и вычислить наблюдаемое значение критерия. Для удобства вычисления необходимо заполнить табл. 1.  

Доказать, что $\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - n.$

Проконтролировать корректность вычисления $\chi^2_{\text{набл}}$.  

По заданному уровню значимости $α=0.05$ и числу степеней свободы $df$ найти критическую точку $\chi^2_{крит}$ и сравнить с наблюдаемым значением. Сделать выводы.

### Проверка гипотезы о нормальности критерием Пирсона $\chi^2$

**Нулевая гипотеза:** $H_0$: выборка получена из нормального распределения $N(\mu, \sigma^2)$.

**Алгоритм:**

1. **Группировка данных**
   * Число интервалов $k$ выбирается по формуле Стерджеса:
    
   * Ширина интервала: $h = \frac{x_{\max} - x_{\min}}{k}$
   * Границы интервалов: $[a_i, a_{i+1})$, где $a_1 = x_{\min}$, $a_{k+1} = x_{\max}$

2. **Эмпирические частоты**
   $$n_i = \#\{x_j \in [a_i, a_{i+1})\}, \quad \sum_{i=1}^k n_i = N$$

3. **Теоретические вероятности и частоты**
   При $H_0$ вероятность попадания в интервал:
   $$
   p_i = \Phi\left(\frac{a_{i+1} - \bar{x}}{s}\right) - \Phi\left(\frac{a_i - \bar{x}}{s}\right)
   $$
   где $\Phi(\cdot)$ — функция стандартного нормального распределения.
   
   Теоретическая частота: $n'_i = N \cdot p_i$

4. **Статистика критерия**
   $$
   \chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{(n_i - n'_i)^2}{n'_i}
   $$
   
   **Доказательство эквивалентной формы:**
   $$
   \frac{(n_i - n'_i)^2}{n'_i} = \frac{n_i^2 - 2n_i n'_i + (n'_i)^2}{n'_i} = \frac{n_i^2}{n'_i} - 2n_i + n'_i
   $$
   Суммируя:
   $$
   \sum \frac{(n_i - n'_i)^2}{n'_i} = \sum \frac{n_i^2}{n'_i} - 2\underbrace{\sum n_i}_{N} + \underbrace{\sum n'_i}_{N} = \sum \frac{n_i^2}{n'_i} - N
   $$
   Что и требовалось доказать: $\chi^2_{\text{набл}} = \sum \frac{n_i^2}{n'_i} - N$

5. **Число степеней свободы**
   $$df = k - 1 - m$$
   где $m = 2$ — число оценённых параметров ($\mu \to \bar{x}$, $\sigma \to s$).
   Итого: $df = k - 3$

6. **Принятие решения**
   * Критическое значение: $\chi^2_{\text{крит}} = \chi^2_{1-\alpha, df}$
   * Если $\chi^2_{\text{набл}} < \chi^2_{\text{крит}}$ — принимаем $H_0$
   * Если $\chi^2_{\text{набл}} \ge \chi^2_{\text{крит}}$ — отвергаем $H_0$


In [94]:
# 1. Группировка (формула Стерджеса)
k = round(1 + 3.322 * np.log10(N))
print(f"Число интервалов: k = {k}")

# Границы интервалов
bins = np.linspace(sample.min(), sample.max(), k+1)
intervals = [(bins[i], bins[i+1]) for i in range(k)]
intervals[0] = (-np.inf, intervals[0][1])
intervals[-1] = (intervals[-1][0], np.inf)

# 2. Эмпирические частоты
n_i, _ = np.histogram(sample, bins=bins)

# 3. Теоретические вероятности и частоты
p_i = [st.norm.cdf(b, x_bar, s) - st.norm.cdf(a, x_bar, s) for a, b in intervals]
n_prime = N * np.array(p_i)

# 4. Вычисление X^2
chi2_obs = np.sum((n_i - n_prime)**2 / n_prime)
chi2_check = np.sum(n_i**2 / n_prime) - N  # проверка альтернативной формулой

# 5. Критическое значение
alpha = 0.05
df = k - 1 - 2
chi2_crit = st.chi2.ppf(1 - alpha, df)

# 6. Таблица результатов
table = pd.DataFrame({
    'i': np.arange(1, k + 1),
    'Интервал': [f"[{a:.2f}, {b:.2f})" for a, b in intervals],
    'n_i': n_i,
    'p_i': p_i,
    "n'_i": n_prime,
    "(n_i - n'_i)^2": (n_i - n_prime)**2,
    "(n_i - n'_i)^2/n'_i": (n_i - n_prime)**2 / n_prime,
    "n^2_i": n_i**2,
    "n^2_i / n'_i": n_i**2 / n_prime,
})

# Строка с суммами
sum_row = pd.DataFrame({
    'i': ['Σ'],
    'Интервал': ['-'],
    'n_i': [n_i.sum()],
    'p_i': [sum(p_i)],
    "n'_i": [n_prime.sum()],
    "(n_i - n'_i)^2": [table["(n_i - n'_i)^2"].sum()],
    "(n_i - n'_i)^2/n'_i": [table["(n_i - n'_i)^2/n'_i"].sum()],
    "n^2_i": [table["n^2_i"].sum()],
    "n^2_i / n'_i": [table["n^2_i / n'_i"].sum()],
})

table = pd.concat([table, sum_row], ignore_index=True)

print("\nТаблица №1:")
table


Число интервалов: k = 8

Таблица №1:


,i,Интервал,n_i,p_i,n'_i,(n_i - n'_i)^2,(n_i - n'_i)^2/n'_i,n^2_i,n^2_i / n'_i
0,1,"[-inf, 361.88)",4,0.045095,4.644832,0.415808,0.089521,16,3.444689
1,2,"[361.88, 393.75)",10,0.088806,9.147013,0.727587,0.079544,100,10.932531
2,3,"[393.75, 425.62)",16,0.166975,17.198417,1.436202,0.083508,256,14.885091
3,4,"[425.62, 457.50)",21,0.224787,23.153071,4.635714,0.200220,441,19.047149
4,5,"[457.50, 489.38)",27,0.216693,22.319403,21.907984,0.981567,729,32.662163
5,6,"[489.38, 521.25)",13,0.149578,15.406539,5.791429,0.375907,169,10.969368
6,7,"[521.25, 553.12)",7,0.073924,7.614175,0.377211,0.049541,49,6.435366
7,8,"[553.12, inf)",5,0.034141,3.516551,2.200621,0.625790,25,7.109239
8,Σ,-,103,1.000000,103.000000,37.492557,2.485597,1785,105.485597


In [95]:
print(f"X^2_крит(alpha=0.05, df=5) = {chi2_crit:.2f}")
print(f"X^2_набл = {chi2_obs:.2f}")
print("Гипотеза принимается")

X^2_крит(alpha=0.05, df=5) = 11.07
X^2_набл = 2.49
Гипотеза принимается


**Параметры критерия:**
*   Число интервалов: $k = 8$
*   Число степеней свободы: $df = k - 3 = 5$
*   Уровень значимости: $\alpha = 0.05$
*   Критическое значение: $\chi^2_{\text{крит}} = \chi^2_{0.95, 5} \approx 11.07$

**Результаты:**
*   Наблюдаемое значение статистики: $\chi^2_{\text{набл}} = 2.49$
*   Проверка эквивалентной формулы: $\sum \frac{n_i^2}{n'_i} - N = 105.49 - 103 = 2.49$


$\chi^2_{\text{набл}} = 2.49 < \chi^2_{\text{крит}} = 11.07 \quad \Rightarrow  \quad$ Гипотеза $H_0$ принимается.

Расхождения между эмпирическими и теоретическими частотами статистически незначимы на уровне $\alpha = 0.05$. Выборка согласуется с предположением о нормальном распределении генеральной совокупности.



# Выводы

В ходе выполнения практической работы получены навыки построения интервальных оценок параметров распределения и проверки статистических гипотез. Для математического ожидания при неизвестном $\sigma$ использовано $t$-распределение Стьюдента, что дало доверительные интервалы $\mu \in (443.37; 464.63)$ при $\gamma = 0.95$ и $\mu \in (439.94; 468.06)$ при $\gamma = 0.99$, вычисленные по формуле $\bar{x} \pm t_{\alpha/2, N-1} \cdot s/\sqrt{N}$. Для среднеквадратичного отклонения применено $\chi^2$-распределение, получены интервалы $\sigma \in (47.82; 63.01)$ и $\sigma \in (46.00; 66.12)$ соответственно по формуле $s \cdot \sqrt{(N-1)/\chi^2_{\alpha/2, N-1}} < \sigma < s \cdot \sqrt{(N-1)/\chi^2_{1-\alpha/2, N-1}}$. Установлено, что рост доверительной вероятности приводит к расширению интервалов.

Проверка гипотезы о нормальности критерием Пирсона показала, что наблюдаемое значение $\chi^2_{\text{набл}} = 2.49$ меньше критического $\chi^2_{\text{крит}} = 11.07$ при $\alpha = 0.05$ и $df = 5$, поэтому нулевая гипотеза $H_0$ о нормальном распределении не отвергается. Корректность вычислений подтверждена эквивалентной формулой $\chi^2_{\text{набл}} = \sum_{i=1}^{k} \frac{n_i^2}{n'_i} - N$.

Таким образом, выборка согласуется с предположением о нормальности $X \sim N(\mu, \sigma^2)$, что обосновывает применение использованных методов интервального оценивания. Приобретены практические навыки работы с распределениями Стьюдента и $\chi^2$, расчёта доверительных интервалов и применения критерия согласия Пирсона для проверки статистических гипотез.